# Mangrove above-ground carbon by Forces of Nature patch

This notebook associates `mangrove_abg_jamaica.tif` with the Forces of Nature mangrove polygons so the above-ground carbon raster can be analysed by mangrove patch.

Workflow:
1. Load the Forces of Nature mangrove polygons.
2. Reproject the above-ground carbon raster from geographic coordinates to Jamaica's metric CRS (`EPSG:3448`).
3. Run patch-level zonal statistics.
4. Convert raster carbon density to estimated total carbon per patch.
5. Flag and map patches without valid raster cells.
6. Create a nearest-neighbour sensitivity fill for no-coverage patches.
7. Save patch-level outputs for later joins with flood, encroachment, or priority-ranking outputs.

Carbon-unit assumption: the raster values are treated as above-ground carbon density in `Mg C ha⁻¹`. Under that assumption, total patch carbon is calculated as `sum(pixel density values) × pixel area in hectares`. If the source metadata later confirms a different unit, update `carbon_density_units` and the total-carbon calculation before using the outputs in reporting.

## Imports

In [ ]:
import os
import sys
from pathlib import Path

base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
matplotlib_config_dir = base_path / ".matplotlib"
matplotlib_config_dir.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(matplotlib_config_dir))

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from IPython.display import display
from matplotlib.patches import Patch
from rasterio.enums import Resampling
from rasterio.errors import WindowError
from rasterio.features import geometry_mask, geometry_window
from rasterio.warp import calculate_default_transform, reproject
from shapely.geometry import LineString

robyn_libraries_path = (base_path / "robyns_libraries").resolve()
if str(robyn_libraries_path) not in sys.path:
    sys.path.append(str(robyn_libraries_path))
import Robyn_paper_2_defs

pd.set_option("display.max_columns", 200)

## Paths and settings

In [ ]:
jamaica_metric_grid_crs = "EPSG:3448"
carbon_density_units = "Mg C ha-1"
include_all_touched_pixels = False

mangrove_abg_raster_path = base_path / "dphil_paper_3/inputs/carbon/mangrove_carbon/mangrove_abg_jamaica.tif"
forces_of_nature_mangrove_path = base_path / "dphil_paper_3/inputs/forces_of_nature_mangroves/mangroves.shp"
mangrove_priority_table_path = base_path / "dphil_paper_3/results_coastal_scenario_comparison/weighted_area_distance_signed/mangrove_priority_ranking/mangrove_priority_full_table_weighted_area_distance.csv"

output_dir = base_path / "dphil_paper_3/processed_data/carbon/mangrove_carbon"
figure_dir = base_path / "dphil_paper_3/results/co_benefits/carbon"
output_dir.mkdir(parents=True, exist_ok=True)
figure_dir.mkdir(parents=True, exist_ok=True)

reprojected_abg_raster_path = output_dir / "mangrove_abg_jamaica_epsg3448.tif"
patch_summary_csv_path = output_dir / "fn_mangrove_patch_abg_carbon_summary.csv"
patch_summary_gpkg_path = output_dir / "fn_mangrove_patch_abg_carbon_summary.gpkg"
requested_patch_table_path = output_dir / "fn_mangrove_patch_abg_carbon_zero_and_nearest_neighbor_table.csv"
nearest_neighbor_summary_csv_path = output_dir / "fn_mangrove_patch_abg_carbon_nearest_neighbor_fill.csv"
nearest_neighbor_summary_gpkg_path = output_dir / "fn_mangrove_patch_abg_carbon_nearest_neighbor_fill.gpkg"
nearest_neighbor_links_gpkg_path = output_dir / "fn_mangrove_patch_abg_carbon_nearest_neighbor_links.gpkg"
parish_summary_csv_path = output_dir / "fn_mangrove_patch_abg_carbon_by_parish.csv"
patch_map_path = figure_dir / "fn_mangrove_patch_abg_carbon_map.png"
coverage_status_map_path = figure_dir / "fn_mangrove_patch_abg_coverage_status_map.png"
zero_coverage_only_map_path = figure_dir / "fn_mangrove_patch_abg_no_valid_cells_only_map.png"
nearest_neighbor_map_path = figure_dir / "fn_mangrove_patch_abg_nearest_neighbor_fill_map.png"
nearest_neighbor_source_map_path = figure_dir / "fn_mangrove_patch_abg_nearest_neighbor_sources_map.png"

for required_path in [mangrove_abg_raster_path, forces_of_nature_mangrove_path, mangrove_priority_table_path]:
    if not required_path.exists():
        raise FileNotFoundError(f"Missing required input: {required_path}")

print(f"Carbon raster: {mangrove_abg_raster_path}")
print(f"Forces of Nature mangroves: {forces_of_nature_mangrove_path}")
print(f"Outputs: {output_dir}")

## Load Forces of Nature mangrove patches

In [ ]:
fn_mangroves = gpd.read_file(forces_of_nature_mangrove_path).to_crs(jamaica_metric_grid_crs)
fn_mangroves = fn_mangroves[fn_mangroves.geometry.notna() & ~fn_mangroves.geometry.is_empty].copy()
fn_mangroves["geometry"] = fn_mangroves.geometry.make_valid()

if fn_mangroves["ID"].isna().any():
    raise ValueError("The Forces of Nature mangrove layer has missing patch IDs in the ID field.")

fn_mangroves["Mangrove_ID"] = fn_mangroves["ID"].astype(int)
fn_mangroves["patch_area_m2"] = fn_mangroves.geometry.area
fn_mangroves["patch_area_ha"] = fn_mangroves["patch_area_m2"] / 10_000

print(f"Mangrove patches: {len(fn_mangroves):,}")
print(f"CRS: {fn_mangroves.crs}")
display(fn_mangroves[["Mangrove_ID", "Parish", "TYPE", "HECTARES", "patch_area_ha"]].head())

## Reproject carbon raster to the Jamaica metric grid

The carbon raster is in `EPSG:4326`, while the Forces of Nature mangrove patches are in `EPSG:3448`. Reprojecting the raster to the metric grid gives a constant pixel area in square metres, which is needed to convert carbon density to total patch carbon.

In [ ]:
with rasterio.open(mangrove_abg_raster_path) as carbon_source:
    source_nodata = carbon_source.nodata
    target_transform, target_width, target_height = calculate_default_transform(
        carbon_source.crs,
        jamaica_metric_grid_crs,
        carbon_source.width,
        carbon_source.height,
        *carbon_source.bounds,
    )
    pixel_area_ha = abs(target_transform.a * target_transform.e) / 10_000

    target_profile = carbon_source.profile.copy()
    target_profile.update(
        {
            "crs": jamaica_metric_grid_crs,
            "transform": target_transform,
            "width": target_width,
            "height": target_height,
            "nodata": source_nodata,
            "compress": "lzw",
        }
    )

    with rasterio.open(reprojected_abg_raster_path, "w", **target_profile) as carbon_destination:
        reproject(
            source=rasterio.band(carbon_source, 1),
            destination=rasterio.band(carbon_destination, 1),
            src_transform=carbon_source.transform,
            src_crs=carbon_source.crs,
            src_nodata=source_nodata,
            dst_transform=target_transform,
            dst_crs=jamaica_metric_grid_crs,
            dst_nodata=source_nodata,
            resampling=Resampling.bilinear,
        )

    raster_metadata = pd.DataFrame(
        [
            {"property": "source_crs", "value": str(carbon_source.crs)},
            {"property": "target_crs", "value": jamaica_metric_grid_crs},
            {"property": "source_shape", "value": f"{carbon_source.height:,} rows × {carbon_source.width:,} columns"},
            {"property": "target_shape", "value": f"{target_height:,} rows × {target_width:,} columns"},
            {"property": "target_pixel_area_ha", "value": pixel_area_ha},
            {"property": "nodata", "value": source_nodata},
            {"property": "density_units_assumed", "value": carbon_density_units},
        ]
    )

display(raster_metadata)
print(f"Saved reprojected raster: {reprojected_abg_raster_path}")

## Calculate patch-level zonal statistics

The custom rasterio helper below assigns raster cells to polygons using the selected pixel rule. The default here is `include_all_touched_pixels = False`, meaning a pixel is counted when its centre falls inside a patch. Set it to `True` above if you want a more inclusive sensitivity run for very small patches.

In [ ]:
def calculate_patch_zonal_statistics(mangrove_patches, raster_path, nodata_value, all_touched):
    patch_rows = []

    with rasterio.open(raster_path) as carbon_raster:
        for mangrove_patch in mangrove_patches[["Mangrove_ID", "geometry"]].itertuples(index=False):
            try:
                patch_window = geometry_window(carbon_raster, [mangrove_patch.geometry])
            except WindowError:
                patch_rows.append(
                    {
                        "valid_abg_pixel_count": 0,
                        "min_abg_carbon_mg_c_ha": np.nan,
                        "mean_abg_carbon_mg_c_ha": np.nan,
                        "max_abg_carbon_mg_c_ha": np.nan,
                        "sum_abg_density_values": np.nan,
                    }
                )
                continue

            raster_values = carbon_raster.read(1, window=patch_window, masked=False)
            patch_mask = geometry_mask(
                [mangrove_patch.geometry],
                out_shape=raster_values.shape,
                transform=carbon_raster.window_transform(patch_window),
                invert=True,
                all_touched=all_touched,
            )
            valid_pixel_mask = patch_mask & np.isfinite(raster_values)
            if nodata_value is not None:
                valid_pixel_mask = valid_pixel_mask & (raster_values != nodata_value)

            patch_values = raster_values[valid_pixel_mask]
            if patch_values.size == 0:
                patch_rows.append(
                    {
                        "valid_abg_pixel_count": 0,
                        "min_abg_carbon_mg_c_ha": np.nan,
                        "mean_abg_carbon_mg_c_ha": np.nan,
                        "max_abg_carbon_mg_c_ha": np.nan,
                        "sum_abg_density_values": np.nan,
                    }
                )
            else:
                patch_rows.append(
                    {
                        "valid_abg_pixel_count": int(patch_values.size),
                        "min_abg_carbon_mg_c_ha": float(patch_values.min()),
                        "mean_abg_carbon_mg_c_ha": float(patch_values.mean()),
                        "max_abg_carbon_mg_c_ha": float(patch_values.max()),
                        "sum_abg_density_values": float(patch_values.sum()),
                    }
                )

    return pd.DataFrame(patch_rows)


carbon_statistics = calculate_patch_zonal_statistics(
    fn_mangroves,
    reprojected_abg_raster_path,
    source_nodata,
    include_all_touched_pixels,
)

carbon_summary = fn_mangroves[
    [
        "Mangrove_ID",
        "Parish",
        "TYPE",
        "TARGET",
        "CONDITION",
        "GEOCLIMATI",
        "HECTARES",
        "patch_area_m2",
        "patch_area_ha",
        "geometry",
    ]
].join(carbon_statistics)

carbon_summary["valid_abg_pixel_count"] = carbon_summary["valid_abg_pixel_count"].fillna(0).astype(int)
carbon_summary["has_observed_abg_carbon"] = carbon_summary["valid_abg_pixel_count"] > 0
carbon_summary["abg_carbon_status"] = np.where(
    carbon_summary["has_observed_abg_carbon"],
    "Observed raster overlap",
    "No valid raster cells",
)
carbon_summary["coverage_area_ha"] = carbon_summary["valid_abg_pixel_count"] * pixel_area_ha
carbon_summary["coverage_pct_of_patch"] = np.where(
    carbon_summary["patch_area_ha"] > 0,
    carbon_summary["coverage_area_ha"] / carbon_summary["patch_area_ha"] * 100,
    np.nan,
)
carbon_summary["total_abg_carbon_mg_c"] = carbon_summary["sum_abg_density_values"] * pixel_area_ha
carbon_summary["total_abg_carbon_tco2e"] = carbon_summary["total_abg_carbon_mg_c"] * (44 / 12)
carbon_summary["abg_carbon_mg_c_per_patch_ha"] = np.where(
    carbon_summary["patch_area_ha"] > 0,
    carbon_summary["total_abg_carbon_mg_c"] / carbon_summary["patch_area_ha"],
    np.nan,
)

display(
    carbon_summary.sort_values("total_abg_carbon_mg_c", ascending=False)[
        [
            "Mangrove_ID",
            "Parish",
            "TYPE",
            "patch_area_ha",
            "valid_abg_pixel_count",
            "coverage_pct_of_patch",
            "mean_abg_carbon_mg_c_ha",
            "total_abg_carbon_mg_c",
            "total_abg_carbon_tco2e",
        ]
    ].head(20)
)

## Nearest-neighbour sensitivity for patches without raster coverage

The primary result keeps no-coverage patches as missing because the raster has no valid non-nodata cells inside those polygons. This sensitivity run fills those patches using the mean above-ground carbon density of the nearest Forces of Nature mangrove patch that does have valid raster coverage. The imputed total is `nearest observed patch mean density × missing patch area`.

This is useful for scenario testing, but it should not replace the observed-raster result unless you are comfortable assuming nearby mapped mangrove patches have comparable above-ground carbon density.

In [ ]:
observed_carbon_patches = carbon_summary[carbon_summary["has_observed_abg_carbon"]].copy()
missing_carbon_patches = carbon_summary[~carbon_summary["has_observed_abg_carbon"]].copy()

if observed_carbon_patches.empty:
    raise ValueError("No Forces of Nature mangrove patches have observed carbon raster coverage.")

nearest_neighbor_rows = []
for missing_patch in missing_carbon_patches.itertuples(index=False):
    source_distances_m = observed_carbon_patches.geometry.distance(missing_patch.geometry)
    source_candidates = pd.DataFrame(
        {
            "source_index": source_distances_m.index,
            "nearest_source_distance_m": source_distances_m.to_numpy(),
            "nearest_source_mangrove_id": observed_carbon_patches["Mangrove_ID"].to_numpy(),
        }
    ).sort_values(["nearest_source_distance_m", "nearest_source_mangrove_id"])
    nearest_source = observed_carbon_patches.loc[source_candidates.iloc[0]["source_index"]]
    nearest_density_mg_c_ha = nearest_source["mean_abg_carbon_mg_c_ha"]

    nearest_neighbor_rows.append(
        {
            "Mangrove_ID": missing_patch.Mangrove_ID,
            "nearest_source_mangrove_id": int(nearest_source["Mangrove_ID"]),
            "nearest_source_parish": nearest_source["Parish"],
            "nearest_source_type": nearest_source["TYPE"],
            "nearest_source_distance_m": float(source_candidates.iloc[0]["nearest_source_distance_m"]),
            "nearest_source_mean_abg_carbon_mg_c_ha": float(nearest_density_mg_c_ha),
            "nn_total_abg_carbon_mg_c": float(nearest_density_mg_c_ha * missing_patch.patch_area_ha),
            "nn_total_abg_carbon_tco2e": float(nearest_density_mg_c_ha * missing_patch.patch_area_ha * (44 / 12)),
        }
    )

nearest_neighbor_fill = pd.DataFrame(nearest_neighbor_rows)
carbon_summary = carbon_summary.merge(nearest_neighbor_fill, on="Mangrove_ID", how="left")
carbon_summary["nearest_source_distance_km"] = carbon_summary["nearest_source_distance_m"] / 1_000
carbon_summary["carbon_estimate_method"] = np.where(
    carbon_summary["has_observed_abg_carbon"],
    "observed_raster_overlap",
    "nearest_observed_patch_mean",
)
carbon_summary["mean_abg_carbon_mg_c_ha_with_nn_fill"] = carbon_summary["mean_abg_carbon_mg_c_ha"].where(
    carbon_summary["has_observed_abg_carbon"],
    carbon_summary["nearest_source_mean_abg_carbon_mg_c_ha"],
)
carbon_summary["total_abg_carbon_mg_c_with_nn_fill"] = carbon_summary["total_abg_carbon_mg_c"].where(
    carbon_summary["has_observed_abg_carbon"],
    carbon_summary["nn_total_abg_carbon_mg_c"],
)
carbon_summary["total_abg_carbon_tco2e_with_nn_fill"] = carbon_summary[
    "total_abg_carbon_mg_c_with_nn_fill"
] * (44 / 12)

nearest_neighbor_fill_gdf = carbon_summary[~carbon_summary["has_observed_abg_carbon"]].copy()
nearest_neighbor_fill_columns = [
    "Mangrove_ID",
    "Parish",
    "TYPE",
    "patch_area_ha",
    "nearest_source_mangrove_id",
    "nearest_source_parish",
    "nearest_source_type",
    "nearest_source_distance_m",
    "nearest_source_distance_km",
    "nearest_source_mean_abg_carbon_mg_c_ha",
    "nn_total_abg_carbon_mg_c",
    "nn_total_abg_carbon_tco2e",
]

nearest_neighbor_source_lookup = observed_carbon_patches.set_index("Mangrove_ID")
nearest_neighbor_link_rows = []
for filled_patch in nearest_neighbor_fill_gdf.itertuples(index=False):
    source_patch = nearest_neighbor_source_lookup.loc[int(filled_patch.nearest_source_mangrove_id)]
    nearest_neighbor_link_rows.append(
        {
            "Mangrove_ID": filled_patch.Mangrove_ID,
            "nearest_source_mangrove_id": int(filled_patch.nearest_source_mangrove_id),
            "nearest_source_distance_m": filled_patch.nearest_source_distance_m,
            "nearest_source_distance_km": filled_patch.nearest_source_distance_km,
            "geometry": LineString(
                [
                    filled_patch.geometry.representative_point(),
                    source_patch.geometry.representative_point(),
                ]
            ),
        }
    )
nearest_neighbor_links = gpd.GeoDataFrame(nearest_neighbor_link_rows, crs=carbon_summary.crs)

patch_positive_avoided_ead = pd.read_csv(
    mangrove_priority_table_path,
    usecols=["Mangrove_ID", "avoided_usd_min", "avoided_usd_max", "avoided_usd_mean"],
).rename(
    columns={
        "avoided_usd_min": "avoided_ead_usd_min",
        "avoided_usd_max": "avoided_ead_usd_max",
        "avoided_usd_mean": "patch_positive_avoided_EADs",
    }
)
carbon_summary = carbon_summary.merge(patch_positive_avoided_ead, on="Mangrove_ID", how="left")

mangrove_carbon_value_mg_c = carbon_summary["total_abg_carbon_mg_c"].fillna(0)
nearest_neighbor_carbon_value_mg_c = carbon_summary["nn_total_abg_carbon_mg_c"].where(
    ~carbon_summary["has_observed_abg_carbon"]
)
requested_patch_table = pd.DataFrame(
    {
        "mangrove_patch_id": carbon_summary["Mangrove_ID"].astype(int),
        "patch_size_hectares": carbon_summary["patch_area_ha"],
        "mangrove_carbon_value_mg_c": mangrove_carbon_value_mg_c,
        "mangrove_carbon_value_mg_per_hectare": np.where(
            carbon_summary["patch_area_ha"] > 0,
            mangrove_carbon_value_mg_c / carbon_summary["patch_area_ha"],
            np.nan,
        ),
        "distance_to_nearest_neighbour_km_for_zero_data": carbon_summary["nearest_source_distance_km"].where(
            ~carbon_summary["has_observed_abg_carbon"]
        ),
        "carbon_value_from_nearest_neighbour_mg_c_for_zero_data": nearest_neighbor_carbon_value_mg_c,
        "carbon_value_from_nearest_neighbour_per_hectare": np.where(
            carbon_summary["patch_area_ha"] > 0,
            nearest_neighbor_carbon_value_mg_c / carbon_summary["patch_area_ha"],
            np.nan,
        ),
        "patch_positive_avoided_EADs": carbon_summary["patch_positive_avoided_EADs"],
        "avoided_ead_usd_min": carbon_summary["avoided_ead_usd_min"],
        "avoided_ead_usd_max": carbon_summary["avoided_ead_usd_max"],
    }
).sort_values("mangrove_patch_id")
requested_patch_table_decimal_columns = [
    column for column in requested_patch_table.columns if column != "mangrove_patch_id"
]
requested_patch_table[requested_patch_table_decimal_columns] = requested_patch_table[
    requested_patch_table_decimal_columns
].round(2)

method_summary = pd.DataFrame(
    [
        {
            "metric": "patches_with_observed_raster_coverage",
            "value": int(carbon_summary["has_observed_abg_carbon"].sum()),
        },
        {
            "metric": "patches_filled_by_nearest_neighbor",
            "value": len(nearest_neighbor_fill_gdf),
        },
        {
            "metric": "observed_total_abg_carbon_mg_c",
            "value": carbon_summary["total_abg_carbon_mg_c"].sum(skipna=True),
        },
        {
            "metric": "nearest_neighbor_filled_total_abg_carbon_mg_c",
            "value": carbon_summary["total_abg_carbon_mg_c_with_nn_fill"].sum(skipna=True),
        },
    ]
)

display(method_summary)
display(requested_patch_table.head(20))
display(
    nearest_neighbor_fill_gdf[nearest_neighbor_fill_columns]
    .sort_values("nearest_source_distance_m", ascending=False)
    .head(20)
)

## Summarise by parish

In [ ]:
parish_summary = (
    carbon_summary.groupby("Parish", dropna=False)
    .agg(
        patch_count=("Mangrove_ID", "count"),
        observed_patch_count=("has_observed_abg_carbon", "sum"),
        patch_area_ha=("patch_area_ha", "sum"),
        total_abg_carbon_mg_c=("total_abg_carbon_mg_c", "sum"),
        total_abg_carbon_tco2e=("total_abg_carbon_tco2e", "sum"),
        total_abg_carbon_mg_c_with_nn_fill=("total_abg_carbon_mg_c_with_nn_fill", "sum"),
        total_abg_carbon_tco2e_with_nn_fill=("total_abg_carbon_tco2e_with_nn_fill", "sum"),
        mean_abg_carbon_mg_c_ha=("mean_abg_carbon_mg_c_ha", "mean"),
        mean_abg_carbon_mg_c_ha_with_nn_fill=("mean_abg_carbon_mg_c_ha_with_nn_fill", "mean"),
    )
    .reset_index()
)
parish_summary["abg_carbon_mg_c_per_patch_ha"] = np.where(
    parish_summary["patch_area_ha"] > 0,
    parish_summary["total_abg_carbon_mg_c"] / parish_summary["patch_area_ha"],
    np.nan,
)
parish_summary["filled_patch_count"] = parish_summary["patch_count"] - parish_summary["observed_patch_count"]
parish_summary["abg_carbon_mg_c_per_patch_ha_with_nn_fill"] = np.where(
    parish_summary["patch_area_ha"] > 0,
    parish_summary["total_abg_carbon_mg_c_with_nn_fill"] / parish_summary["patch_area_ha"],
    np.nan,
)
parish_summary = parish_summary.sort_values("total_abg_carbon_mg_c", ascending=False)

display(parish_summary)

## Save patch and parish outputs

In [ ]:
patch_summary_columns = [column for column in carbon_summary.columns if column != "geometry"]
carbon_summary[patch_summary_columns].to_csv(patch_summary_csv_path, index=False)
carbon_summary.to_file(patch_summary_gpkg_path, driver="GPKG")
requested_patch_table.to_csv(requested_patch_table_path, index=False, float_format="%.2f")
nearest_neighbor_fill_gdf[nearest_neighbor_fill_columns].to_csv(nearest_neighbor_summary_csv_path, index=False)
if len(nearest_neighbor_fill_gdf) > 0:
    nearest_neighbor_fill_gdf.to_file(nearest_neighbor_summary_gpkg_path, driver="GPKG")
if len(nearest_neighbor_links) > 0:
    nearest_neighbor_links.to_file(nearest_neighbor_links_gpkg_path, driver="GPKG")
parish_summary.to_csv(parish_summary_csv_path, index=False)

print(f"Saved patch CSV: {patch_summary_csv_path}")
print(f"Saved patch GeoPackage: {patch_summary_gpkg_path}")
print(f"Saved requested patch table: {requested_patch_table_path}")
print(f"Saved nearest-neighbour CSV: {nearest_neighbor_summary_csv_path}")
if len(nearest_neighbor_fill_gdf) > 0:
    print(f"Saved nearest-neighbour GeoPackage: {nearest_neighbor_summary_gpkg_path}")
if len(nearest_neighbor_links) > 0:
    print(f"Saved nearest-neighbour links: {nearest_neighbor_links_gpkg_path}")
print(f"Saved parish CSV: {parish_summary_csv_path}")

## Map patch-level carbon totals

In [ ]:
figure, axis = plt.subplots(figsize=(8, 10))
carbon_summary.plot(
    ax=axis,
    column="total_abg_carbon_mg_c",
    cmap="YlGn",
    legend=True,
    legend_kwds={"shrink": 0.55, "label": "Total AGB carbon (Mg C)"},
    linewidth=0.15,
    edgecolor="black",
    missing_kwds={"color": "lightgrey", "label": "No raster cells"},
)

axis.set_title("Above-ground mangrove carbon by Forces of Nature patch")
axis.set_axis_off()
Robyn_paper_2_defs.draw_scale_bar(
    axis,
    location=(0.88, 0.78),
    length_km=20,
    linewidth=0.6,
    label_offset=0.02,
    km_offset=0.01,
)
Robyn_paper_2_defs.draw_north_arrow(axis, location=(0.88, 0.86), size=0.05, fontsize=8, label_offset=0.02)

figure.tight_layout()
figure.savefig(patch_map_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved map: {patch_map_path}")

## Map raster-coverage status

This map highlights the Forces of Nature mangrove patches with no valid above-ground-carbon raster cells. These are the patches filled in the nearest-neighbour sensitivity output.

In [ ]:
coverage_status_colors = {
    "Observed raster overlap": "#BDBDBD",
    "No valid raster cells": "#C62828",
}

figure, axis = plt.subplots(figsize=(8, 10))
for coverage_status, coverage_color in coverage_status_colors.items():
    coverage_subset = carbon_summary[carbon_summary["abg_carbon_status"] == coverage_status]
    if len(coverage_subset) > 0:
        coverage_subset.plot(ax=axis, color=coverage_color, edgecolor="black", linewidth=0.15)

missing_coverage_patches = carbon_summary[~carbon_summary["has_observed_abg_carbon"]].copy()
if len(missing_coverage_patches) > 0:
    missing_coverage_points = missing_coverage_patches.set_geometry(missing_coverage_patches.representative_point())
    missing_coverage_points.plot(
        ax=axis,
        color="#C62828",
        edgecolor="white",
        linewidth=0.3,
        markersize=20,
        marker="o",
    )

axis.set_title("Forces of Nature mangrove patches without valid AGB raster cells")
axis.set_axis_off()
Robyn_paper_2_defs.draw_scale_bar(
    axis,
    location=(0.88, 0.78),
    length_km=20,
    linewidth=0.6,
    label_offset=0.02,
    km_offset=0.01,
)
Robyn_paper_2_defs.draw_north_arrow(axis, location=(0.88, 0.86), size=0.05, fontsize=8, label_offset=0.02)
coverage_legend_handles = [
    Patch(facecolor=coverage_status_colors[coverage_status], edgecolor="black", label=coverage_status)
    for coverage_status in coverage_status_colors
]
axis.legend(handles=coverage_legend_handles, loc="lower left", frameon=True)

figure.tight_layout()
figure.savefig(coverage_status_map_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved coverage-status map: {coverage_status_map_path}")

## Map only patches with no valid AGB raster cells

This map removes the observed patches and labels only the Forces of Nature mangrove patches where the above-ground-carbon raster contributes zero valid cells.

In [ ]:
zero_coverage_patches = carbon_summary[~carbon_summary["has_observed_abg_carbon"]].copy()
zero_coverage_points = zero_coverage_patches.set_geometry(zero_coverage_patches.representative_point())

figure, axis = plt.subplots(figsize=(9, 10))
zero_coverage_patches.plot(ax=axis, color="#C62828", edgecolor="black", linewidth=0.25)
zero_coverage_points.plot(
    ax=axis,
    color="#C62828",
    edgecolor="white",
    linewidth=0.4,
    markersize=28,
    marker="o",
)

for zero_patch in zero_coverage_points.itertuples(index=False):
    axis.annotate(
        str(int(zero_patch.Mangrove_ID)),
        xy=(zero_patch.geometry.x, zero_patch.geometry.y),
        xytext=(3, 3),
        textcoords="offset points",
        fontsize=5,
        color="black",
    )

axis.set_title("Forces of Nature mangrove patches with zero valid AGB raster cells")
axis.set_axis_off()
Robyn_paper_2_defs.draw_scale_bar(
    axis,
    location=(0.88, 0.78),
    length_km=20,
    linewidth=0.6,
    label_offset=0.02,
    km_offset=0.01,
)
Robyn_paper_2_defs.draw_north_arrow(axis, location=(0.88, 0.86), size=0.05, fontsize=8, label_offset=0.02)
axis.legend(
    handles=[Patch(facecolor="#C62828", edgecolor="black", label="No valid AGB raster cells")],
    loc="lower left",
    frameon=True,
)

figure.tight_layout()
figure.savefig(zero_coverage_only_map_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved zero-coverage-only map: {zero_coverage_only_map_path}")

## Map nearest-neighbour filled carbon estimates

This map uses observed raster totals where available and nearest-neighbour filled totals for no-coverage patches. Red outlines mark patches where the carbon value is imputed rather than observed from the raster.

In [ ]:
figure, axis = plt.subplots(figsize=(8, 10))
carbon_summary.plot(
    ax=axis,
    column="total_abg_carbon_mg_c_with_nn_fill",
    cmap="YlGn",
    legend=True,
    legend_kwds={"shrink": 0.55, "label": "Total AGB carbon (Mg C)"},
    linewidth=0.15,
    edgecolor="black",
)
if len(nearest_neighbor_fill_gdf) > 0:
    nearest_neighbor_fill_gdf.boundary.plot(ax=axis, color="#C62828", linewidth=0.8)

axis.set_title("Above-ground mangrove carbon with nearest-neighbour fill")
axis.set_axis_off()
Robyn_paper_2_defs.draw_scale_bar(
    axis,
    location=(0.88, 0.78),
    length_km=20,
    linewidth=0.6,
    label_offset=0.02,
    km_offset=0.01,
)
Robyn_paper_2_defs.draw_north_arrow(axis, location=(0.88, 0.86), size=0.05, fontsize=8, label_offset=0.02)
nearest_neighbor_legend_handles = [
    Patch(facecolor="none", edgecolor="#C62828", label="Nearest-neighbour filled")
]
axis.legend(handles=nearest_neighbor_legend_handles, loc="lower left", frameon=True)

figure.tight_layout()
figure.savefig(nearest_neighbor_map_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved nearest-neighbour map: {nearest_neighbor_map_path}")

## Map nearest-neighbour source patches

This map shows the no-coverage patches, the observed patches used as nearest-neighbour sources, and the source-to-fill links. It is intended as a QA map for deciding whether the nearest-neighbour fill is defensible.

In [ ]:
nearest_source_ids = nearest_neighbor_fill_gdf["nearest_source_mangrove_id"].dropna().astype(int).unique()
nearest_source_patches = carbon_summary[carbon_summary["Mangrove_ID"].isin(nearest_source_ids)].copy()
nearest_source_points = nearest_source_patches.set_geometry(nearest_source_patches.representative_point())
nearest_fill_points = nearest_neighbor_fill_gdf.set_geometry(nearest_neighbor_fill_gdf.representative_point())

figure, axis = plt.subplots(figsize=(9, 10))
nearest_neighbor_links.plot(ax=axis, color="#757575", linewidth=0.35, alpha=0.7)
nearest_source_patches.plot(ax=axis, color="#1565C0", edgecolor="black", linewidth=0.2, alpha=0.75)
nearest_neighbor_fill_gdf.plot(ax=axis, color="#C62828", edgecolor="black", linewidth=0.25, alpha=0.9)
nearest_source_points.plot(
    ax=axis,
    color="#1565C0",
    edgecolor="white",
    linewidth=0.3,
    markersize=20,
    marker="o",
)
nearest_fill_points.plot(
    ax=axis,
    color="#C62828",
    edgecolor="white",
    linewidth=0.3,
    markersize=20,
    marker="o",
)

axis.set_title("Nearest observed AGB source patches for no-coverage FN mangroves")
axis.set_axis_off()
Robyn_paper_2_defs.draw_scale_bar(
    axis,
    location=(0.88, 0.78),
    length_km=20,
    linewidth=0.6,
    label_offset=0.02,
    km_offset=0.01,
)
Robyn_paper_2_defs.draw_north_arrow(axis, location=(0.88, 0.86), size=0.05, fontsize=8, label_offset=0.02)
source_link_legend_handles = [
    Patch(facecolor="#C62828", edgecolor="black", label="No valid AGB cells / filled patch"),
    Patch(facecolor="#1565C0", edgecolor="black", label="Nearest observed source patch"),
    Patch(facecolor="#757575", edgecolor="#757575", label="Nearest-neighbour link"),
]
axis.legend(handles=source_link_legend_handles, loc="lower left", frameon=True)

figure.tight_layout()
figure.savefig(nearest_neighbor_source_map_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved nearest-neighbour source map: {nearest_neighbor_source_map_path}")

## How to use this with other mangrove-patch analyses

Use `Mangrove_ID` as the join key. The source Forces of Nature shapefile stores the patch identifier in `ID`; this notebook converts it to integer `Mangrove_ID` to match the coastal-flooding and encroachment workflows.

Suggested joins:
- Join `fn_mangrove_patch_abg_carbon_summary.csv` to the mangrove priority table on `Mangrove_ID`.
- Use `total_abg_carbon_mg_c` for total stock comparisons.
- Use `total_abg_carbon_mg_c_with_nn_fill` when you want no-coverage patches filled from the nearest observed coastal mangrove patch.
- Use `carbon_estimate_method` to separate observed raster values from nearest-neighbour filled values.
- Review `fn_mangrove_patch_abg_carbon_nearest_neighbor_links.gpkg` and the source-link map before final reporting.
- Use `abg_carbon_mg_c_per_patch_ha` or `mean_abg_carbon_mg_c_ha` for density-style comparisons.
- Check `coverage_pct_of_patch` and `abg_carbon_status` before interpreting small or no-coverage patches.